# 🏥 ClinixVault GPU Backend & Cloudflare Tunnel on Google Colab

This notebook turns a free or Pro Google Colab instance (T4, L4, or A100 GPU) into a high-throughput **ClinixVault API Gateway & Inference Node**.

### ✨ Features
- 🚀 **Zero-Account Cloudflare Tunnel**: Generates a public HTTPS URL with native **WebSocket** support for real-time Ambient Scribe audio streaming.
- ⚡ **Faster-Whisper `large-v3-turbo` on CUDA**: High-accuracy speech recognition with VRAM safety protection.
- 📁 **Dual Model Sourcing**: Instant boot from mounted **Google Drive** (zero bandwidth) or automatic download from **Hugging Face Hub**.
- 📊 **Grafana Cloud Telemetry**: Real-time GPU VRAM, compute utilization, request latencies, and logs streamed to Grafana Cloud.
- 🔄 **Live Code Sync**: Pull latest codebase changes from Git without losing session state or restarting the kernel.

### Step 1: Verify GPU & Clone/Sync Repository

In [ ]:
# Check CUDA GPU availability
!nvidia-smi

# Clone or Sync ClinixVault repository
import os
REPO_URL = "https://github.com/samarthbhandegaonkar/clinixvault.git"  # Update with your repo URL
BRANCH = "main"

if not os.path.exists("/content/clinixvault"):
    print("Cloning ClinixVault repository...")
    !git clone --depth 1 -b {BRANCH} {REPO_URL} /content/clinixvault
else:
    print("Syncing existing repository...")
    %cd /content/clinixvault
    !git pull origin {BRANCH}

%cd /content/clinixvault

### Step 2: Fast Optimized Dependency Installation
Installs pre-built wheels with caching to spin up the backend in under 60 seconds.

In [ ]:
%%time
# Fast install of critical inference dependencies
!pip install --quiet --upgrade pip
!pip install --quiet fastapi uvicorn[standard] faster-whisper huggingface_hub peft transformers pydantic sqlalchemy psycopg2-binary psutil
print("✅ Core backend dependencies installed successfully.")

### Step 3: Choose Model Sourcing Strategy
- **Option A (`drive`)**: Instant boot from mounted Google Drive (Zero download time or data transfer).
- **Option B (`huggingface`)**: Automatic fetch from Hugging Face Hub (cached in `/content/cache/hf`).

In [ ]:
#@title Model Source Configuration
MODEL_SOURCE = "huggingface" #@param ["drive", "huggingface"]
DRIVE_MODELS_PATH = "/content/drive/MyDrive/clinixvault_models" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}
HF_ORG = "ClinixVault" #@param {type:"string"}

if MODEL_SOURCE == "drive":
    from google.colab import drive
    print("📁 Mounting Google Drive...")
    drive.mount('/content/drive')
    print(f"✓ Google Drive mounted at /content/drive. Looking for models in: {DRIVE_MODELS_PATH}")
else:
    print(f"🌐 Configured to fetch fine-tuned weights from Hugging Face Hub ({HF_ORG})")

### Step 4: Optional Grafana Cloud Observability Configuration
Inject your Grafana Cloud credentials to push real-time GPU VRAM, CPU, and API metrics directly to your dashboard (`livelycelery2196.grafana.net`).

In [ ]:
#@title Grafana Cloud Credentials (Optional)
GRAFANA_CLOUD_PROMETHEUS_URL = "https://prometheus-prod-43-prod-ap-south-1.grafana.net/api/prom/push" #@param {type:"string"}
GRAFANA_CLOUD_PROMETHEUS_USER = "3688243" #@param {type:"string"}
GRAFANA_CLOUD_LOKI_URL = "https://logs-prod-028.grafana.net/loki/api/v1/push" #@param {type:"string"}
GRAFANA_CLOUD_LOKI_USER = "1839786" #@param {type:"string"}
GRAFANA_CLOUD_API_TOKEN = "" #@param {type:"string"}

if GRAFANA_CLOUD_API_TOKEN:
    os.environ["GRAFANA_CLOUD_PROMETHEUS_URL"] = GRAFANA_CLOUD_PROMETHEUS_URL
    os.environ["GRAFANA_CLOUD_PROMETHEUS_USER"] = GRAFANA_CLOUD_PROMETHEUS_USER
    os.environ["GRAFANA_CLOUD_LOKI_URL"] = GRAFANA_CLOUD_LOKI_URL
    os.environ["GRAFANA_CLOUD_LOKI_USER"] = GRAFANA_CLOUD_LOKI_USER
    os.environ["GRAFANA_CLOUD_API_TOKEN"] = GRAFANA_CLOUD_API_TOKEN
    print("📊 Grafana Cloud credentials configured for Colab telemetry push.")
else:
    print("ℹ️  Grafana Cloud token omitted. Local /metrics endpoint will still expose OpenMetrics.")

### Step 5: Start ClinixVault GPU Backend & Cloudflare Tunnel
Runs the FastAPI server with CUDA GPU acceleration and outputs the public URL.

In [ ]:
# Start Colab Runner with GPU VRAM Watchdog & Cloudflare Tunnel
!python colab/colab_runner.py \
    --model-source {MODEL_SOURCE} \
    --drive-path "{DRIVE_MODELS_PATH}" \
    --hf-token "{HF_TOKEN}" \
    --hf-org "{HF_ORG}" \
    --whisper-model "large-v3-turbo"

### Step 6: Code Sync Cell (Hot Reload Changes from Git)
Whenever you modify code locally and push to your git repository, run this cell to sync the latest Python files without restarting the Colab runtime!

In [ ]:
# Run code sync
!python colab/colab_runner.py --sync-git